# 0. Experiment Contract
**Bilateral five-epoch fine-tune, 80GB profile (not hardware-validated).** The unchanged model is two MaxViT 2D encoders plus ResNeXt3D and CrossGate. Only `bilateral_s42` is trained, from the pinned `raw_s42_recovered_20260910/raw_s42/best_weights.pt`. No archived raw start, pretrained download, random-weight fallback, or emergency checkpoint substitution. Real settings: raw storage/3D 200, 2D 224, batch 2, accumulation 8, LR 5e-5, weight decay 1e-4, five epochs, patience 5, checkpoint every 10 optimizer steps. RESUME uses a safe last.pt when present; an identity stranded before its first checkpoint recovers the same W&B ID with a fresh optimizer and verified parent weights. An entirely new RESUME is rejected.
The pinned Bilateral export is imported from the authenticated HF dataset `tqhuyen/harvard-oct-glaucoma-200-bilateral` (section 4) instead of refiltering; `CPU_EXPORT_ROOT` points at the downloaded export and raw volumes are still hash-verified against it. Imported arrays are published to the shared Drive cache (`PUBLISH_DATA_CACHE`) before training.
Stages 3-6 are CPU preparation. Real preflight, training and inference are separately opt-in. Preflight uses the full model, pinned weights, two optimizer windows (16 microbatches), resident gradients and allocated optimizer state. Its time estimate is approximate, not a benchmark or extra training epochs. CUDA >=70 GiB and sufficient free memory are mandatory; the budget is 0.85. BF16 is selected when supported, otherwise the float16 fallback is recorded. No compile or activation checkpointing.
**Independent rerender:** in a fresh runtime run sections 1-2, then 10 and optionally 13. Saved validation/test logits and provenance suffice; no datasets, Trainer, fit, model or GPU are needed. Section 9 can resume split predictions independently. Optional XAI (11) and information theory (12) use separate linked W&B runs and cannot invalidate analysis completion, including when their own publication fails. Unlike the generic three-branch notebook, this fixed five-epoch study intentionally forbids epoch extension and evaluates held-out test only after training.
First SIGINT/SIGTERM requests a safe optimizer-boundary stop; a second interrupt may produce separate non-resumable emergency weights. Inspect `fx.ACTIVE_TRAINER` / `fx.ACTIVE_MODEL` after failure; call `fx.release_active()` before reconstructing with RESUME. Never resume emergency weights. Checkpoints retain deterministic sample order, RNG, partial metrics and best state. Same GPU/software/data are required; CUDA bitwise reproducibility is not promised.
`FINAL_SMOKE=1` explicitly allows a tiny synthetic CPU model/parent and offline W&B. Actual Bilateral preprocessing, verified simulated-Drive copies, all stages and bounded XAI execute. Smoke is NOT validation of real CUDA, online W&B or mounted Drive. Never edit a running external kernel's settings expecting it to change.

## 1. Setup, Environment And Repository Hashes
This notebook pins normalized LF SHA256 for eight reviewed helpers, including information_theory, before imports. Missing/dirty/stale code or unverified imported modules require a fresh reviewed clone/runtime; existing clones are never reset or pulled automatically. After publication, FINAL_CODE_REVISION selects the reviewed commit and FINAL_REPO_ROOT a fresh location. timm==1.0.29 is pinned and CPU construction is tested. The merged fuse() method preserves the old forward computation and state-dict keys; no encoder, g3 normalization or architecture is changed. Actual parent loading and the 80GB profile remain unvalidated until the target-host probe passes.

In [ ]:
import hashlib
import importlib
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

SMOKE = os.environ.get("FINAL_SMOKE", "0") == "1"
REVIEWED_REVISION = os.environ.get("FINAL_CODE_REVISION")
if REVIEWED_REVISION and (len(REVIEWED_REVISION) != 40 or any(c not in "0123456789abcdef" for c in REVIEWED_REVISION)):
    raise ValueError("FINAL_CODE_REVISION must be a full lowercase 40-character Git SHA")
if not SMOKE and "google.colab" in sys.modules:
    repo = Path(os.environ.get("FINAL_REPO_ROOT", "/content/glaucoma-thesis"))
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Tqhuyen/glaucoma-thesis.git", str(repo)], check=True
        )
        if REVIEWED_REVISION:
            subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", REVIEWED_REVISION], check=True)
            subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
    if (
        REVIEWED_REVISION
        and subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
        != REVIEWED_REVISION
    ):
        raise RuntimeError(
            "Existing clone has another revision. Preserve it and set FINAL_REPO_ROOT to a fresh location; no reset/pull performed."
        )
    os.chdir(repo)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "timm==1.0.29",
            "wandb",
            "scikit-image",
            "scikit-learn",
            "scipy",
            "matplotlib",
            "huggingface_hub",
            "python-dotenv",
            "psutil",
            "hf-transfer",
        ],
        check=True,
    )
sys.path.insert(0, str(Path.cwd()))
if not SMOKE:
    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
EXPECTED_HELPER_SHA256 = {
    "final_model.py": "62ca10c93a0d20211ccd804e968a20aec690fc911735ed3149e885d3e102f053",
    "final_training.py": "ac2baf10c50c0cd92a51d545ce077123bfdaf36d0406008290731966fedf5363",
    "final_data.py": "f01384e47df7c1d6f2cd3d08b64ce8b1d177a9e73aa4a0305c681c746c6dfd07",
    "final_reporting.py": "8266394e382bff0687ccedc157d038f693f8164cd194995d714a817acb5ceeae",
    "final_execution.py": "f4cf5f7ffe2fae4680e09404b0115f12bc1895deccd116a859a711d909593c67",
    "resolution_study.py": "32ce1bfbf22e246a052e306e6f5e3e3690a732f26771db08b6ec5364bc5b50ca",
    "compare_denoise_methods.py": "53091aa5df3e450609c504c46870778430ef3ca363767614c78ec02a0f72600a",
    "information_theory.py": "0b688850c3f49146ce3c7d766646d9cb1fb0eaed22c988e158cb2983b0c69d05",
}
EXPECTED_BUNDLE_SHA256 = "45b84ab310ed82abc13b435751671fcf6159c2373e9ae7e84fb97687ade341ed"
REPO_ROOT = Path.cwd().resolve()


def verify_reviewed_helpers():
    package = sys.modules.get("scripts")
    if package is not None and REPO_ROOT / "scripts" not in [
        Path(p).resolve() for p in getattr(package, "__path__", [])
    ]:
        raise RuntimeError(
            "scripts package belongs to another clone; restart the runtime with the reviewed repository."
        )
    for filename, expected in EXPECTED_HELPER_SHA256.items():
        path = REPO_ROOT / "scripts" / filename
        actual = (
            hashlib.sha256(
                path.read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n").encode()
            ).hexdigest()
            if path.is_file()
            else None
        )
        if actual != expected:
            raise RuntimeError(
                f"Reviewed helper missing/dirty/stale: {path}. Preserve this clone; use a fresh clone of the published reviewed revision with this notebook. No automatic pull/reset is performed."
            )
    if (
        hashlib.sha256(json.dumps(EXPECTED_HELPER_SHA256, sort_keys=True, allow_nan=False).encode()).hexdigest()
        != EXPECTED_BUNDLE_SHA256
    ):
        raise RuntimeError("Notebook helper bundle digest is inconsistent; obtain the reviewed notebook.")
    for name, module in list(sys.modules.items()):
        filename = name.rsplit(".", 1)[-1] + ".py"
        if filename in EXPECTED_HELPER_SHA256 and module is not None:
            if (
                Path(getattr(module, "__file__", "")).resolve() != REPO_ROOT / "scripts" / filename
                or getattr(module, "_FINAL_REVIEWED_SOURCE_SHA256", None) != EXPECTED_HELPER_SHA256[filename]
            ):
                raise RuntimeError(
                    f"Unverified/stale imported helper {name}; restart the runtime before running section 1. Never reload an active training model."
                )


verify_reviewed_helpers()
for filename in EXPECTED_HELPER_SHA256:
    importlib.import_module("scripts." + filename[:-3])
for name, module in list(sys.modules.items()):
    filename = name.rsplit(".", 1)[-1] + ".py"
    if filename in EXPECTED_HELPER_SHA256 and module is not None:
        if Path(module.__file__).resolve() != REPO_ROOT / "scripts" / filename:
            raise RuntimeError(f"Imported helper path mismatch: {name}; restart with the reviewed clone.")
        module._FINAL_REVIEWED_SOURCE_SHA256 = EXPECTED_HELPER_SHA256[filename]
verify_reviewed_helpers()
import matplotlib
import torch

matplotlib.use("Agg")
from scripts import final_execution as fx
from scripts import final_training as ft

ft.load_env_file()
if SMOKE:
    torch.set_num_threads(1)
CODE_IDENTITY = fx.code_identity()
print(CODE_IDENTITY)

## 2. Configuration, Drive And W&B Checks
Run group is separate from the old schema. Set `FINAL_LOCAL_ROOT` to restore the same workspace in another runtime. The remote group restores `context.pt` and individual stage files. The shared `data_cache/bilateral` is NOT nested under a run. Mount and online W&B checks fail loudly. Flags are deliberately excluded from the immutable training identity. For rerender only, leave all GPU flags off.

In [ ]:
# ===== RUN IDENTITY & RESUME (edit only when starting a new experiment) =====
RUN_GROUP = "bilateral_s42_finetune5_80gb_v1"
RUN_TARGET = "bilateral_s42"
RESUME = False
EXTEND_EPOCHS = 0
# ===== STAGE SWITCHES (enable only the stages you intend to run) =====
NUM_WORKERS = 0
STEP_METRICS = True
ENABLE_GPU_PREFLIGHT = SMOKE
ENABLE_TRAIN = SMOKE
ENABLE_EVAL = SMOKE
ALLOW_BUILD_DENOISED = SMOKE
RUN_XAI = SMOKE
RUN_INFO = SMOKE
INFO_OPTIONS = dict(
    subset=8 if SMOKE else 128,
    pca_dim=2 if SMOKE else 8,
    input_res=2 if SMOKE else 4,
    steps=2 if SMOKE else 100,
    probe_steps=2 if SMOKE else 100,
    hidden=8 if SMOKE else 32,
    seeds=[0, 1] if SMOKE else [0, 1, 2, 3, 4],
    surrogates=2 if SMOKE else 1000,
    mine_surrogates=2 if SMOKE else 5,
)
# ===== DATA SOURCES & IMPORT (HF repos, import/publish cache) =====
DENOISE_LIMIT = 0
CACHE_MANIFEST = None
CPU_EXPORT_ROOT = None
HF_DN_REPO = "tqhuyen/harvard-oct-glaucoma-200-bilateral"
HF_DN_REVISION = "47632c96b206707fd6423ee5b4da159069f63eaf"
HF_DN_LOCAL_DIR = "/content/final_dn_export"
PUBLISH_DATA_CACHE = SMOKE
HF_RAW_REVISION = None
# ===== FROZEN STUDY CONFIG (validated for this study; do not retune) =====
STORE_RES = 8 if SMOKE else 200
RES3D, RES2D = (8, 8) if SMOKE else (200, 224)
EPOCHS, BS, GRAD_ACCUM = (1, 2, 2) if SMOKE else (5, 2, 8)
LR, WD, PATIENCE = 5e-5, 1e-4, 5
CHECKPOINT_EVERY_STEPS = 10
CHECKPOINT_EVERY_SECONDS = 300
FUSED_ADAMW = False
PREFLIGHT_WINDOWS, PREFLIGHT_FRACTION = 2, 0.85
# ===== WARM-START PARENT (pinned raw weights; do not substitute) =====
WARM_START_WEIGHTS = "/content/drive/MyDrive/MasterBKDN/Thesis/final_2x2d_3d_crossgate/raw_s42_recovered_20260910/raw_s42/best_weights.pt"
# ===== DERIVED PATHS & DRIVE (computed from the above; do not edit) =====
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix="final_smoke_")) if SMOKE else None
DRIVE_MOUNT = Path(os.environ.get("DRIVE_MOUNT", "/content/drive"))
DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/MasterBKDN/Thesis"))
DRIVE_DIR = SMOKE_ROOT / "simulated_drive" / RUN_GROUP if SMOKE else DRIVE_ROOT / "final_2x2d_3d_crossgate" / RUN_GROUP
LOCAL_ROOT = Path(
    os.environ.get(
        "FINAL_LOCAL_ROOT", str(SMOKE_ROOT / "run" if SMOKE else Path("outputs/final_2x2d_3d_crossgate") / RUN_GROUP)
    )
)
DATA_ROOT = SMOKE_ROOT / "data" if SMOKE else Path("/content/final_data")
CACHE_ROOT = (
    SMOKE_ROOT / "simulated_drive" / "data_cache" / "bilateral"
    if SMOKE
    else DRIVE_ROOT / "final_2x2d_3d_crossgate" / "data_cache" / "bilateral"
)
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive

        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError("Drive must be a verified mounted filesystem")
DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
if SMOKE:
    WARM_START_WEIGHTS = str(SMOKE_ROOT / "parent" / "best_weights.pt")
    fx.smoke_parent(WARM_START_WEIGHTS)
config = dict(
    dataset="bilateral",
    seed=42,
    epochs=EPOCHS,
    batch_size=BS,
    grad_accum=GRAD_ACCUM,
    lr=LR,
    weight_decay=WD,
    patience=PATIENCE,
    checkpoint_steps=CHECKPOINT_EVERY_STEPS,
    checkpoint_seconds=CHECKPOINT_EVERY_SECONDS,
    store_res=STORE_RES,
    res3d=RES3D,
    res2d=RES2D,
    n2d=2,
    latent=256,
    enc2d="maxvit_tiny_rw_224",
    tested_timm_version="1.0.29",
    smoke=SMOKE,
    amp_dtype="float16",
    pin_memory=not SMOKE,
    non_blocking=not SMOKE,
    fused_adamw=FUSED_ADAMW,
    telemetry=True,
    step_metrics=STEP_METRICS,
    num_workers=NUM_WORKERS,
    extend_epochs=EXTEND_EPOCHS,
    max_scaler_skips=3,
    preflight_windows=PREFLIGHT_WINDOWS,
    preflight_fraction=PREFLIGHT_FRACTION,
    training_phase="bilateral_finetune5_80gb_v1",
    compile=False,
    activation_checkpointing=False,
)
context = dict(
    config=config,
    smoke=SMOKE,
    remote=str(DRIVE_DIR),
    data_root=str(DATA_ROOT),
    cache_root=str(CACHE_ROOT),
    parent=WARM_START_WEIGHTS,
)
STORAGE = fx.workspace(LOCAL_ROOT, DRIVE_DIR, smoke=SMOKE, context=context)
STORAGE.save(CODE_IDENTITY, "code_identity.pt")
check_artifacts, WANDB_RUN = fx.stage_run(LOCAL_ROOT, "checks", config, "not-started")
fx.finish_stage(check_artifacts, WANDB_RUN, {"purpose": "storage and W&B connectivity", "smoke": SMOKE})
print("Workspace:", LOCAL_ROOT, "Drive:", DRIVE_DIR, "Shared cache:", CACHE_ROOT)


## 3. Parent Verification (CPU)
Verify pinned path, raw seed 42/dimensions/architecture, run identity, last.pt and metrics/history, best epoch/AUC and tensor-exact best weights. Missing old completed.pt records a missing completion phase; it does not force raw retraining or claim completion. No emergency fallback.

In [ ]:
print(fx.parent_stage(LOCAL_ROOT))

## 4. Raw Download (CPU)
Compare all six raw/label hashes with the parent before any filtering. Missing files resolve HF once to a pinned 40-SHA and persist the download plan. Existing matching bytes without revision evidence are adopted as parent_hashes_only, NOT assigned an HF revision. Set CPU_EXPORT_ROOT (or HF_RAW_REVISION) to verify existing bytes against that revision's HF LFS SHA256 / Git blobs and write trusted source_manifest.json. CPU export import needs this evidence; it never requires refiltering. No downsampled arrays are persisted.

In [ ]:
if SMOKE:
    print("Smoke: synthetic build path retained; no HF bilateral export download.")
else:
    from huggingface_hub import snapshot_download

    ft.load_env_file()
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata

            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        raise RuntimeError("HF_TOKEN is required for the authenticated bilateral export download")
    if (
        not HF_DN_REPO
        or len(HF_DN_REVISION) != 40
        or any(c not in "0123456789abcdef" for c in HF_DN_REVISION)
    ):
        raise RuntimeError("Set HF_DN_REPO and a pinned 40-SHA HF_DN_REVISION")
    DN_EXPORT_ROOT = Path(HF_DN_LOCAL_DIR)
    snapshot_download(
        repo_id=HF_DN_REPO,
        repo_type="dataset",
        revision=HF_DN_REVISION,
        local_dir=str(DN_EXPORT_ROOT),
        token=token,
        allow_patterns=["manifest.json", "README.md", "*_complete.json", "*_volumes_dn.npy", "*_labels.npy"],
    )
    exported = json.loads((DN_EXPORT_ROOT / "manifest.json").read_text(encoding="utf-8"))
    if exported.get("complete") is not True:
        raise RuntimeError("Downloaded bilateral export is not complete")
    if exported["identity"]["source_repo"] != fx.RAW_REPO:
        raise RuntimeError("Bilateral export source repository mismatch")
    revision = exported["identity"]["source_revision"]
    if len(revision) != 40 or any(c not in "0123456789abcdef" for c in revision):
        raise RuntimeError("Bilateral export source revision is not a pinned 40-SHA")
    CPU_EXPORT_ROOT = str(DN_EXPORT_ROOT)
    print("Bilateral export ready:", CPU_EXPORT_ROOT)
fx.raw_stage(LOCAL_ROOT, revision=HF_RAW_REVISION, cpu_export_root=CPU_EXPORT_ROOT)


## 5. Bilateral Restore Or Preparation (CPU)
Select CACHE_MANIFEST for a portable producer cache or CPU_EXPORT_ROOT for the completed prepare_bilateral_200 export (downloaded from HF in section 3b), not both. Producer filter/provenance are preserved across consumer Torch versions; CPU import never calls the current cdm denoiser. ALLOW_BUILD_DENOISED=False and PUBLISH_DATA_CACHE=False for real runs. Bulk publication requires explicit consent; locally prepared/imported arrays remain pending until verified in the shared Drive cache. Set PUBLISH_DATA_CACHE=True and rerun this section to upload existing arrays without refiltering. DENOISE_LIMIT bounds explicit builds. Smoke explicitly builds and publishes only tiny synthetic Bilateral data.

In [ ]:
fx.prepare_stage(
    LOCAL_ROOT,
    allow_build=ALLOW_BUILD_DENOISED,
    limit=DENOISE_LIMIT,
    manifest_path=CACHE_MANIFEST,
    cpu_export_root=CPU_EXPORT_ROOT,
    publish_cache=PUBLISH_DATA_CACHE,
)

## 6. Datasets And Full Source Manifest Audit
Recheck raw, labels, denoised volumes, 2D views and depth-axis SHA256 identities for all splits. Save execution.pt as the explicit CPU-to-GPU handoff. Both branches and all splits use Bilateral sources; no raw fallback. num_workers=0 preserves deterministic resume.

In [ ]:
config = fx.audit_data_stage(LOCAL_ROOT)
print("Audited splits:", list(config["data"]["splits"]))

## 7. GPU80 Preflight (Opt-In)
Enable once on the target GPU. The corrected ft.find_batch_size uses ONLY candidate [2] for this preset, loads the full pinned model, and runs two optimizer windows including resident optimizer state and accumulated gradients. Batch/accumulation never change. Buffer/weight/mode/RNG restoration and finally cleanup protect callers; no training checkpoint is modified. report/batch_probe_table records the budget and trials. The passed digest binds GPU stack/device, backend, code, parent and data. Resume reuses this saved probe and batch; changed environment requires a new explicit probe. Fused AdamW is opt-in with the same tested backend. No local80 measurement is claimed.

In [ ]:
print(fx.preflight_stage(LOCAL_ROOT, enabled=ENABLE_GPU_PREFLIGHT))

## 8. Training (Opt-In; Only Fit Stage)
ENABLE_TRAIN requires the exact successful probe digest. W&B and JSONL retain every boundary loss/accuracy, system telemetry, timing, and full epoch train/validation metrics. STEP_METRICS=True additionally logs the merged full window metric set; turning it off only suppresses window statistics, never scalar logging. Windows and this exact-resume preset use NUM_WORKERS=0. Generic Trainer supports per-index deterministic worker loaders and explicitly authorized epoch extension, but this final study requires EXTEND_EPOCHS=0 and five epochs. It never evaluates test, XAI or information estimators inside fit.

In [ ]:
print(fx.training_stage(LOCAL_ROOT, enabled=ENABLE_TRAIN, resume=RESUME))

## 9. Independent Evaluation (Opt-In; No Trainer Or Fit)
Load the saved training handoff/best weights. Validation and test logits are persisted immediately after each split. A restart reuses matching cached splits, including on CPU when both exist. No optimizer or Trainer is constructed. W&B has a separate linked evaluation run. Calibration is deferred to section 10.

In [ ]:
print(fx.evaluation_stage(LOCAL_ROOT, enabled=ENABLE_EVAL))

## 10. CPU Report Render (Independent)
Run directly after sections 1-2 in a fresh runtime with existing saved predictions. Validation-only temperature/Youden selection, raw-margin ROC AUC and sklearn AP, full metrics, 0.5 operating points, scan-level bootstrap CIs, CSV/Markdown and figures use final_reporting. No test leakage, GPU, datasets or training state. Missing evaluation completion is reported as pending, not success. Each rerender logs to its own W&B run linked to training.

In [ ]:
_, evaluation_artifacts = fx.open_stage(LOCAL_ROOT, "evaluation")
if (evaluation_artifacts.local / "completed.pt").exists() or (evaluation_artifacts.remote / "completed.pt").exists():
    REPORT = fx.report_stage(LOCAL_ROOT)
    print("Held-out test:", REPORT["test"])
else:
    print("Report pending: complete independent evaluation first; no training invoked.")

## 11. Optional XAI (Off By Default)
Independent stage on validation row 0 only. Real XAI can be expensive and is disabled. Smoke uses bounded Grad-CAM, occlusion and integrated gradients. The first historical branch_drop label actually zeros all 2D inputs, not 3D: outputs rename it all_2d_inputs without changing model semantics. XAI failure leaves all earlier completed stages valid; skip this section to audit/report.

In [ ]:
print(fx.xai_stage(LOCAL_ROOT, enabled=RUN_XAI))

## 12. Optional Information Theory (Off By Default)
RUN_INFO=False for real credit-limited runs. Each cell below has a disk-backed handoff and a separate W&B run linked to training, with verified Drive artifacts. Collection independently restores the best checkpoint and samples bounded, balanced training/validation subsets, never test. Remaining phases run on CPU and reuse saved embeddings without a model or Trainer. Probe ablations, multi-seed DV/NWJ/two-sample InfoNCE, approximate MIC, permutation surrogates, conditional MI and interaction proxies retain negative estimates and log tables/scalars, not figures. PCA is fit on training representations. The information plane is explicitly ONE best-checkpoint snapshot, not a fabricated epoch trajectory. Real defaults: 128 scans/split, PCA 8, 100 estimator/probe steps, 5 seeds, 1000 MIC and 5 MINE surrogates. MINE p-values are coarse; proxies are not PID or patient-level inference.

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="collect", options=INFO_OPTIONS)

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="probes", options=INFO_OPTIONS)

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="estimators", options=INFO_OPTIONS)

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="surrogates", options=INFO_OPTIONS)

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="interactions", options=INFO_OPTIONS)

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="plane", options=INFO_OPTIONS)

In [ ]:
fx.information_stage(LOCAL_ROOT, enabled=RUN_INFO, phase="report", options=INFO_OPTIONS)

## 13. Artifact Audit And Completion
Only a hash-validated final analysis publishes final_summary.pt. All scientific files are SHA-verified on Drive before W&B success finish; completion markers are written afterward. Marker sync failure requires recovery, never a false success. Partial metrics and optional XAI are not summary candidates. Single-seed, scan-level uncertainty does not support patient-level claims.

In [ ]:
_, analysis_artifacts = fx.open_stage(LOCAL_ROOT, "analysis")
if (analysis_artifacts.local / "completed.pt").exists() or (analysis_artifacts.remote / "completed.pt").exists():
    print(fx.completion_stage(LOCAL_ROOT))
else:
    print("Analysis pending; no final summary published.")
print("CPU/offline/simulated-Drive smoke only." if SMOKE else "Only verified completed artifacts are published.")